In [ ]:
from notebook_utils import cd_parent
cd_parent()

from rdkit import Chem
from chemicalgof import Reduce2GoF, GoF2fragSMILES, GoF2Mol, fragSMILES2GoF

import pandas as pd

import multiprocessing as mp


from functools import partial

---

> ⚠️ Set the prefered number of CPU employed for multiprocessing in the following pipeline

---

In [ ]:
# NCPU = mp.cpu_count()
# NCPU = 25

In [3]:
def safe_function(input, fnc, default = None):
	try:
		output = fnc(input)
	except:
		output = default

	return output

In [4]:
dataset = pd.read_csv('data/test.csv', header=None, names=['smiles'])
dataset

,smiles
0,CSc1nn(-c2cccc(C)c2)c2cc(C3=CCNCC3)ccc12
1,CSc1nn(-c2cccc(F)c2)c2cc(C3=CCNCC3)ccc12
2,CSc1nn(-c2ccc(C)cc2)c2cc(C3=CCNCC3)ccc12
3,O=C(c1ccc2c(c1)[nH]c(=O)c1cnn(C3CCCC3)c12)N1CC...
4,Cc1cc2[nH]c(=O)c3cnn(C4CCOCC4)c3c2cc1C(=O)N1CC...
...,...
270403,OC[C@@H]1CCCN1Cc1nc2ccccc2n1Cc1ccc(Cl)cc1
270404,Cc1ccc2nc(CN3CCCC3)n(Cc3ccc(Cl)cc3)c2c1
270405,NC1=NC2(c3cc(-c4cncnc4)ccc3OCC23CC3)C(F)(F)CS1
270406,CS(=O)(=O)Nc1ccc(OC[C@@H](O)CN2CCN(c3ccc(Cl)cc...


In [5]:
with mp.Pool(NCPU) as pool:
    canonized_smiles = pool.map(Chem.CanonSmiles, dataset.smiles)
    
dataset['smiles'] = canonized_smiles

In [6]:
safe_reduce2gof = partial(safe_function, fnc=Reduce2GoF)

with mp.Pool(NCPU) as pool:
    reduced_graphs = pool.map(safe_reduce2gof, dataset.smiles)
    
dataset['gof'] = reduced_graphs

/home/tox/chemicalgof_repositories/chemicalgof-030/chemicalgof/utils.py:103: UserWarning: 
Opposite CIP labels from different Legacy implementations.
CIP labels without Legacy implementation were employed: 6R
  warnings.warn(warning_msg)
/home/tox/chemicalgof_repositories/chemicalgof-030/chemicalgof/utils.py:103: UserWarning: 
Opposite CIP labels from different Legacy implementations.
CIP labels without Legacy implementation were employed: 20R
  warnings.warn(warning_msg)
/home/tox/chemicalgof_repositories/chemicalgof-030/chemicalgof/utils.py:103: UserWarning: 
Opposite CIP labels from different Legacy implementations.
CIP labels without Legacy implementation were employed: 18R
  warnings.warn(warning_msg)
/home/tox/chemicalgof_repositories/chemicalgof-030/chemicalgof/utils.py:103: UserWarning: 
Opposite CIP labels from different Legacy implementations.
CIP labels without Legacy implementation were employed: 25R
  warnings.warn(warning_msg)
/home/tox/chemicalgof_repositories/chemicalgo

In [7]:
GoF2fragSMILES_canonized = partial(GoF2fragSMILES, canonize=True)
save_gof2fragsmiles = partial(safe_function, fnc = GoF2fragSMILES_canonized)

with mp.Pool(NCPU) as pool:
    fragsmiles_strings = pool.map(save_gof2fragsmiles, reduced_graphs)

dataset['fragsmiles'] = fragsmiles_strings

In [8]:
def Gof2canonical_smiles(gof):
    return Chem.CanonSmiles(Chem.MolToSmiles(GoF2Mol(gof)))

safe_gof2canonical_smiles = partial(safe_function, fnc = Gof2canonical_smiles)

with mp.Pool(NCPU) as pool:
    smiles_from_gofs = pool.map(safe_gof2canonical_smiles, reduced_graphs)

dataset['smiles_from_gof'] = smiles_from_gofs

/home/tox/chemicalgof_repositories/chemicalgof-030/chemicalgof/utils.py:62: RuntimeWarning: Could not assign modern CIP labels: 
Digraph generation failed: more than 100000nodes found.
  warnings.warn(
/home/tox/chemicalgof_repositories/chemicalgof-030/chemicalgof/utils.py:62: RuntimeWarning: Could not assign modern CIP labels: 
Digraph generation failed: more than 100000nodes found.
  warnings.warn(


In [9]:
different_smiles_from_gof = dataset['smiles'] != dataset['smiles_from_gof']

nan_smiles_from_gof = dataset['smiles_from_gof'].isna()


In [10]:
(different_smiles_from_gof & ~nan_smiles_from_gof).sum()

np.int64(0)

In [11]:
(different_smiles_from_gof & nan_smiles_from_gof).sum()

np.int64(0)

In [12]:
(~different_smiles_from_gof & ~nan_smiles_from_gof).sum()

np.int64(270408)

In [13]:
(different_smiles_from_gof).sum()

np.int64(0)

In [14]:
dataset[different_smiles_from_gof]

,smiles,gof,fragsmiles,smiles_from_gof


In [15]:
def fragsmiles2canonical_smiles(fragsmiles):
    return Gof2canonical_smiles(fragSMILES2GoF(fragsmiles))

safe_fragsmiles2canonical_smiles = partial(safe_function, fnc=fragsmiles2canonical_smiles)

with mp.Pool(NCPU) as pool:
    smiles_from_fragsmiles_series = pool.map(safe_fragsmiles2canonical_smiles, dataset['fragsmiles'])
    
dataset['smiles_from_fragsmiles'] = smiles_from_fragsmiles_series

/home/tox/chemicalgof_repositories/chemicalgof-030/chemicalgof/utils.py:62: RuntimeWarning: Could not assign modern CIP labels: 
Digraph generation failed: more than 100000nodes found.
  warnings.warn(
/home/tox/chemicalgof_repositories/chemicalgof-030/chemicalgof/utils.py:62: RuntimeWarning: Could not assign modern CIP labels: 
Digraph generation failed: more than 100000nodes found.
  warnings.warn(


In [16]:
different_smiles_from_fragsmiles = dataset['smiles'] != dataset['smiles_from_fragsmiles']

nan_smiles_from_fragsmiles = dataset['smiles_from_fragsmiles'].isna()

In [17]:
dataset[different_smiles_from_fragsmiles]

,smiles,gof,fragsmiles,smiles_from_gof,smiles_from_fragsmiles
